# Collect Reasoning Chains
Collect chains of LLMs doing distractor generation on single math problems:
- simple prompt with non-reasoning model
- chain of thought with non-reasoning model
- simple prompt with reasoning model
- ls-informed prompt with reasoning model

In [ ]:
import os
import json
from tqdm import tqdm
from collections import defaultdict
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI

from src.datasets import ADataset, get_or_create_dataset
from src.equality import SemanticEqualityChecker
from src.models.joint import JointModel
from src.models.impl.naive import DeepseekNaiveJointModel, OpenRouterNaiveJointModel
from src.models.impl.enforce_process import DeepseekEnforceProcessJointModel,OpenRouterEnforceProcessJointModel
from src.models.impl.naive_cot import DeepseekNaiveCoTJointModel, OpenRouterNaiveCoTJointModel
from src.model_configurations import gpt_4_1_mini_det_config, deepseek_chat, openrouter_glm_4_7_chat, deepseek_reasoner, openrouter_glm_4_7_reason, openrouter_gpt_oss_20b_reason

load_dotenv()

True

### Dataset

In [ ]:
equivalence_check_path = "cache/semantic_equivalence_checker.pkl"

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
print(f"We have {len(eedi_dataset)} EEDI questions")

datasets_by_datafolder = {
    "eedi_data": eedi_dataset
}

equality_model_config = gpt_4_1_mini_det_config
equality_client = OpenAI(base_url=equality_model_config["base_url"], api_key=os.environ.get(equality_model_config["api_key_var"], None))
if os.path.exists(equivalence_check_path):
    print("Loading existing semantic equality checker")
    semantic_equality_checker = SemanticEqualityChecker.load(equality_client, equivalence_check_path)
else:
    semantic_equality_checker = SemanticEqualityChecker(equality_client, equality_model_config)
    semantic_equality_checker.save(equivalence_check_path)

### Generate Reasoning Traces

In [ ]:
save_every_n_questions = 10
n_retry_limit = 10

In [ ]:
def run_joint(data_folder: str, dataset: ADataset, setting_name: str, model: JointModel, num_distractors: int):
    results_folder = os.path.join(data_folder, "joint_results")
    os.makedirs(results_folder, exist_ok=True)
    response_file = os.path.join(results_folder, f"{setting_name}_responses_by_datapointid.json")

    responses_by_datapointid = defaultdict(dict)
    if os.path.exists(response_file): 
        with open(response_file, "r+") as f:
            responses_by_datapointid = json.load(f)

    # Helper function to process a single datapoint
    def process_datapoint(i):
        datapoint_id = str(i)
        if datapoint_id in responses_by_datapointid:
            return None  # Already processed
        
        context = dataset[i]
        _, distractors_meta = model.generate_distractors(context, num_distractors=num_distractors)
        return (datapoint_id, distractors_meta)

    # Collect unprocessed indices
    unprocessed_indices = [i for i in range(len(dataset)) if str(i) not in responses_by_datapointid]
    
    if not unprocessed_indices:
        print(f"All datapoints already processed for {setting_name}")
        semantic_equality_checker.save(equivalence_check_path)
        return

    # Process in parallel with checkpointing
    with ThreadPoolExecutor(max_workers=8) as executor:
        # Submit all tasks
        futures = {executor.submit(process_datapoint, i): i for i in unprocessed_indices}
        
        # Process results as they complete, with periodic checkpointing
        completed_count = 0
        for future in tqdm(as_completed(futures), total=len(futures)):
            result = future.result()
            if result is not None:
                datapoint_id, distractors_meta = result
                responses_by_datapointid[datapoint_id] = distractors_meta
                completed_count += 1
                
                # Checkpoint periodically
                if completed_count % save_every_n_questions == 0:
                    with open(response_file, "w+") as f:
                        json.dump(responses_by_datapointid, f)

    # Final save
    with open(response_file, "w+") as f:
        json.dump(responses_by_datapointid, f)

    semantic_equality_checker.save(equivalence_check_path)

In [ ]:
openrouter_client = OpenAI(
    api_key=os.environ[openrouter_glm_4_7_chat.get("api_key_var")],
    base_url="https://openrouter.ai/api/v1"
)

#### Direct

In [ ]:
run_joint("eedi_data", eedi_dataset, f"deepseek-naive-deepseek-chat",
            model=DeepseekNaiveJointModel(deepseek_chat), num_distractors=3)

run_joint("eedi_data", eedi_dataset, f"openrouter-naive-z-ai_glm-4.7-chat",
            model=OpenRouterNaiveJointModel(openrouter_client, openrouter_glm_4_7_chat), num_distractors=3)

#### CoT

In [ ]:
run_joint("eedi_data", eedi_dataset, f"deepseek-naive-cot-deepseek-chat",
          model=DeepseekNaiveCoTJointModel(deepseek_chat), num_distractors=3)

run_joint("eedi_data", eedi_dataset, f"openrouter-naive-cot-z-ai_glm-4.7-chat", 
          model=OpenRouterNaiveCoTJointModel(openrouter_client, openrouter_glm_4_7_chat), num_distractors=3)

#### Reasoning

In [ ]:
run_joint("eedi_data", eedi_dataset, f"deepseek-naive-deepseek-reasoner",
            model=DeepseekNaiveJointModel(deepseek_reasoner), num_distractors=3)

run_joint("eedi_data", eedi_dataset, f"openrouter-naive-z-ai_glm-4.7-reasoner",
            model=OpenRouterNaiveJointModel(openrouter_client, openrouter_glm_4_7_reason), num_distractors=3)

run_joint("eedi_data", eedi_dataset, f"openrouter-naive-gpt-oss-20b-reasoner",
            model=OpenRouterNaiveJointModel(openrouter_client, openrouter_gpt_oss_20b_reason), num_distractors=3)

#### Learning-Science Informed

In [ ]:
run_joint("eedi_data", eedi_dataset, f"deepseek-enforce-process-deepseek-reasoner", 
            model=DeepseekEnforceProcessJointModel(deepseek_reasoner, show_correct=False), num_distractors=3)

run_joint("eedi_data", eedi_dataset, f"openrouter-enforce-process-z-ai_glm-4.7-reasoner", 
            model=OpenRouterEnforceProcessJointModel(openrouter_client, openrouter_glm_4_7_reason, show_correct=False), num_distractors=3)

run_joint("eedi_data", eedi_dataset, f"openrouter-enforce-process-gpt-oss-20b-reasoner", 
            model=OpenRouterEnforceProcessJointModel(openrouter_client, openrouter_gpt_oss_20b_reason, show_correct=False), num_distractors=3)